In [3]:
from xrds_handler import XRDS_handler

In [4]:
xrds = XRDS_handler('./data/tx_ens_mean_0.1deg_reg_2011-2023_v29.0e.nc')

In [ ]:
# heatwave
# at least 3 days long
# has the Tx over the 90th percentile of the control period

# control period is 30 years at least
# on a 31 day moving window

In [5]:
import numpy as np
import pandas as pd

class Heatwave:
    '''
    Determines if there was a heatwave at a specific location.
    Stores in an array the days when it happened.
    '''
    
    def __init__(self, xrds_handler, variable, lat, lon, control_period=('2011-01-01', '2023-12-31'), 
                 percentile=90, day_window_size=31):
        self.xrds_handler = xrds_handler  # Instance of XRDS_handler
        self.variable = variable          # Variable name, e.g., 'Tx' for max temperature
        self.lat = lat                    # Latitude for specific location
        self.lon = lon                    # Longitude for specific location
        self.control_period = control_period  # Control period tuple (start, end)
        self.percentile = percentile          # Percentile for heatwave threshold
        self.day_window_size = day_window_size  # Size of the moving window (e.g., 31 days)
        self.heatwave_days = []            # Stores the days that are part of a heatwave

    def calculate_percentile(self, timeseries):
        '''
        Calculate the rolling 90th percentile based on the day_window_size and control period.
        The rolling window is centered on the current day.
        '''
        # Convert to pandas Series for easier time-based operations
        df = pd.Series(timeseries.values, index=timeseries.time.values)

        # Rolling 31-day window, calculate 90th percentile, centered on the current day
        rolling_percentile = df.rolling(window=self.day_window_size, center=True).quantile(self.percentile / 100)

        return rolling_percentile

    def detect_heatwaves(self, start_date, end_date):
        '''
        Detect heatwave days in the given time range (start_date to end_date).
        A day is part of a heatwave if its value is above the 90th percentile, 
        and heatwaves last for at least 3 consecutive days.
        '''
        # Get the temperature time series for the specific location and range
        temp_series = self.xrds_handler.get_ds_at_spec_latlon(self.variable, self.lat, self.lon, start_date, end_date)
        
        # Get the control period data to calculate percentiles
        control_series = self.xrds_handler.get_ds_at_spec_latlon(self.variable, self.lat, self.lon, 
                                                                 self.control_period[0], self.control_period[1])
        
        # Calculate the 90th percentile over the control period with a moving window
        percentiles = self.calculate_percentile(control_series)

        # Find heatwave days
        heatwave_streak = 0  # To track consecutive days above threshold
        for day in temp_series.time.values:
            temp_on_day = temp_series.sel(time=day).values
            threshold = percentiles.loc[day]

            # If the temp exceeds the 90th percentile threshold
            if temp_on_day > threshold:
                heatwave_streak += 1
                if heatwave_streak >= 3:  # At least 3 consecutive days
                    self.heatwave_days.append(day)
            else:
                heatwave_streak = 0  # Reset the streak if below threshold

        return self.heatwave_days

    def get_heatwave_days(self):
        '''Return the detected heatwave days.'''
        return self.heatwave_days


In [9]:
# Create a Heatwave instance for a specific location and variable
hw = Heatwave(xrds, variable='tx', lat=50, lon=10)

# Detect heatwaves in a given date range (e.g., summer of 2022)
heatwave_days = hw.detect_heatwaves(start_date='2011-06-01', end_date='2023-08-31')

# Print heatwave days
print("Heatwave Days:", heatwave_days)
    

Heatwave Days: [np.datetime64('2011-10-02T00:00:00.000000000'), np.datetime64('2012-01-03T00:00:00.000000000'), np.datetime64('2012-07-27T00:00:00.000000000'), np.datetime64('2012-08-21T00:00:00.000000000'), np.datetime64('2012-12-25T00:00:00.000000000'), np.datetime64('2013-01-31T00:00:00.000000000'), np.datetime64('2013-06-19T00:00:00.000000000'), np.datetime64('2013-09-07T00:00:00.000000000'), np.datetime64('2013-09-26T00:00:00.000000000'), np.datetime64('2014-04-25T00:00:00.000000000'), np.datetime64('2014-05-22T00:00:00.000000000'), np.datetime64('2014-06-10T00:00:00.000000000'), np.datetime64('2014-09-08T00:00:00.000000000'), np.datetime64('2015-11-09T00:00:00.000000000'), np.datetime64('2016-04-05T00:00:00.000000000'), np.datetime64('2016-06-24T00:00:00.000000000'), np.datetime64('2016-09-14T00:00:00.000000000'), np.datetime64('2017-11-24T00:00:00.000000000'), np.datetime64('2018-01-01T00:00:00.000000000'), np.datetime64('2018-03-12T00:00:00.000000000'), np.datetime64('2018-04-2